In [1]:
# Read in the anndata object
import anndata as ad
import pandas as pd
import warnings
import re

warnings.filterwarnings('ignore')

In [2]:
# =============================================================================
# Load the full splicing data
# =============================================================================

# Base directories
BASE_DIR = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION"
original_splice_adata = f"{BASE_DIR}/MODEL_INPUT/102025/model_ready_aligned_splicing_data_20251009_024406.h5ad"
full_splice_adata = ad.read_h5ad(original_splice_adata)

In [3]:
print(f"The number of cells in the full splicing data is {full_splice_adata.n_obs}")

The number of cells in the full splicing data is 142315


### We are going to use the original provided cell types (saved in cell_ontology_class) and tissue labels and then manusally sort them into broad cell type, medium cell type and tissue_celltype labels that we will use for all the figures in the manuscript

In [4]:
cell_types_summary = full_splice_adata.obs[["cell_ontology_class", "tissue"]].value_counts()
cell_types_summary = pd.DataFrame(cell_types_summary)
cell_types_summary.to_csv("cell_types_summary_original_labels.csv")

#### This is now the file we are reading that contains the manually curated labels which we will use for everything downstream

In [5]:
full_splice_adata.obs.drop(columns=["medium_cell_type", "broad_cell_type", "specific_cell_type"], inplace=True)

In [7]:
# 1. Read the file
cell_type_mapping = pd.read_csv("/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/Figures/cell_ontology_tissue_OG_Claude_tissue_cell_type_cleaned_nov29.tsv", sep="\t")

In [8]:
cell_type_mapping[cell_type_mapping["tissue_celltype"] == "Lung_Immune"]

,cell_ontology_class,broad_cell_type,tissue,tissue_celltype,dataset,count
421,leukocyte,Immune,Lung,Lung_Immune,TMS,34
546,lymphocyte,Immune,Lung,Lung_Immune,TMS,8


In [10]:
mapper_df = cell_type_mapping[["cell_ontology_class", "tissue", "broad_cell_type", "tissue_celltype"]].drop_duplicates()
mapper_df

,cell_ontology_class,tissue,broad_cell_type,tissue_celltype
0,microglial cell,Brain,Microglia,Brain_Microglia
1,168_L2/3 IT CTX,Brain,Neuron_Excitatory,Brain_Ex-Ctx
2,fibroblast of cardiac tissue,Heart,Stromal,Heart_Fibroblast
3,hematopoietic stem cell,Marrow,Stem/Progenitor,Marrow_HSC
4,granulocyte,Marrow,Immune,Marrow_Myeloid
...,...,...,...,...
580,127_L2 IT APr,Brain,Neuron_Excitatory,Brain_Ex-Ctx
581,382_SMC-Peri,Brain,Stromal,Brain_Pericyte
582,378_Astro,Brain,Glia,Brain_Astrocyte
583,369_Oligo,Brain,Glia,Brain_Oligo


In [11]:
# Need to rename brain tissue to just be "Brain"
# find all cells in splice_adata.obs that contain "Brain" in the tissue column 
brain_cells = full_splice_adata.obs[full_splice_adata.obs["tissue"].str.contains("Brain")].index
# need to first convert to string
full_splice_adata.obs["tissue"] = full_splice_adata.obs["tissue"].astype(str)
# rename these tissues to just be "Brain"
full_splice_adata.obs.loc[brain_cells, "tissue"] = "Brain"

In [12]:
# now need to merge mapper_df onto full_splice_adata.obs on cell_ontology_class and tissue 
# split into brain and non-brain because if brain, we will merge only on cell_ontology_class, but if non-brain, we will merge on cell_ontology_class and tissue 
full_splice_adata.obs = full_splice_adata.obs.merge(mapper_df, on=["cell_ontology_class", "tissue"], how="left")

In [13]:
full_splice_adata.obs[full_splice_adata.obs["tissue_celltype"] == "Lung_Immune"]

,cell_id_index,age,cell_ontology_class,mouse.id,sex,subtissue,tissue,dataset,cell_name,cell_id,cell_clean,seqtech,library_size,total_junction_reads,annotated_junction_reads,unannotated_junction_reads,n_detected_annotated_junctions,n_detected_unannotated_junctions,broad_cell_type,tissue_celltype
9170,9170,3m,leukocyte,3_39_F,F,EPCAM,Lung,TMS,B22_MAA001847,B22_MAA001847,B22_MAA001847,single_cell,21331,19038,18721,317,3228,67,Immune,Lung_Immune
9897,9897,3m,lymphocyte,3_38_F,F,EPCAM,Lung,TMS,B4_MAA001889,B4_MAA001889,B4_MAA001889,single_cell,240994,220929,216818,4111,1286,63,Immune,Lung_Immune
14038,14038,24m,lymphocyte,24_61_M,M,EPCAM,Lung,TMS,C20_B000342,C20_B000342,C20_B000342,single_cell,41035,37945,37668,277,661,13,Immune,Lung_Immune
18959,18959,24m,leukocyte,24_58_M,M,EPCAM,Lung,TMS,D18_B002946,D18_B002946,D18_B002946,single_cell,29299,26540,25957,583,1094,36,Immune,Lung_Immune
19219,19219,24m,leukocyte,24_58_M,M,EPCAM,Lung,TMS,D19_B002946,D19_B002946,D19_B002946,single_cell,7159,6721,6700,21,365,10,Immune,Lung_Immune
19666,19666,24m,leukocyte,24_58_M,M,EPCAM,Lung,TMS,D20_B002946,D20_B002946,D20_B002946,single_cell,27048,25212,24637,575,762,28,Immune,Lung_Immune
27989,27989,24m,leukocyte,24_61_M,M,EPCAM,Lung,TMS,F11_B000342,F11_B000342,F11_B000342,single_cell,7827,6929,6848,81,592,11,Immune,Lung_Immune
28267,28267,24m,leukocyte,24_61_M,M,EPCAM,Lung,TMS,F12_B000342,F12_B000342,F12_B000342,single_cell,11116,10083,9961,122,757,10,Immune,Lung_Immune
28555,28555,24m,leukocyte,24_61_M,M,EPCAM,Lung,TMS,F13_B000342,F13_B000342,F13_B000342,single_cell,110066,100108,98508,1600,1830,48,Immune,Lung_Immune
28834,28834,24m,leukocyte,24_61_M,M,EPCAM,Lung,TMS,F14_B000342,F14_B000342,F14_B000342,single_cell,106406,97685,94376,3309,1846,60,Immune,Lung_Immune


In [14]:
full_splice_adata.obs[full_splice_adata.obs["tissue"] == "Brain"]

,cell_id_index,age,cell_ontology_class,mouse.id,sex,subtissue,tissue,dataset,cell_name,cell_id,cell_clean,seqtech,library_size,total_junction_reads,annotated_junction_reads,unannotated_junction_reads,n_detected_annotated_junctions,n_detected_unannotated_junctions,broad_cell_type,tissue_celltype
6,6,18m,endothelial cell,18_53_M,M,Cortex,Brain,TMS,A10_B000176,A10_B000176,A10_B000176,single_cell,128355,115089,113768,1321,782,25,Endothelial,Brain_Endothelial
43,43,3m,microglial cell,3_38_F,F,Striatum,Brain,TMS,A10_B000825,A10_B000825,A10_B000825,single_cell,42923,37640,36995,645,477,26,Microglia,Brain_Microglia
47,47,24m,microglial cell,24_60_M,M,Striatum,Brain,TMS,A10_B000840,A10_B000840,A10_B000840,single_cell,63555,57927,56177,1750,732,25,Microglia,Brain_Microglia
48,48,24m,microglial cell,24_60_M,M,Hippocampus,Brain,TMS,A10_B000843,A10_B000843,A10_B000843,single_cell,58419,53702,52909,793,513,18,Microglia,Brain_Microglia
54,54,18m,microglial cell,18_47_F,F,Cerebellum,Brain,TMS,A10_B001060,A10_B001060,A10_B001060,single_cell,39178,36024,35389,635,1061,47,Microglia,Brain_Microglia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142310,142310,2m,197_L5 IT CTX,184750,F,VISp,Brain,AB,US-1250275_E2_S86,US-1250275_E2_S86,SRR16457632,single_nuclei,53587,48275,45749,2526,9393,240,Neuron_Excitatory,Brain_Ex-Ctx
142311,142311,2m,204_L5/6 IT CTX,184756,M,VISp,Brain,AB,US-1250275_E2_S87,US-1250275_E2_S87,SRR16457633,single_nuclei,44150,39687,36238,3449,8268,218,Neuron_Excitatory,Brain_Ex-Ctx
142312,142312,2m,257_L5 PT CTX,185199,M,VISp,Brain,AB,US-1250275_E2_S88,US-1250275_E2_S88,SRR16457635,single_nuclei,48161,42726,39975,2751,10177,232,Neuron_Excitatory,Brain_Ex-Ctx
142313,142313,2m,204_L5/6 IT CTX,185200,F,VISp,Brain,AB,US-1250275_E2_S89,US-1250275_E2_S89,SRR16457636,single_nuclei,73787,66108,62267,3841,11739,365,Neuron_Excitatory,Brain_Ex-Ctx


In [16]:
# use this to make supp table 1 
file_name = "LeafletFA_Supplemental_Table_1.xlsx"
full_splice_adata.obs[["cell_id", "cell_name", "cell_ontology_class", "broad_cell_type", "tissue", "tissue_celltype", "dataset"]].drop_duplicates().to_excel(file_name, index=False)

In [17]:
!pwd

/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/Figures
